In [11]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [12]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [13]:
print(len(documents))

72


In [14]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [15]:
results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [16]:
for r in results:
    print(r["filename"])

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/16-other-frameworks.md


In [17]:
def search(query):
    return index.search(
        query=query,
        num_results=5
    )

In [18]:
def build_context(search_results):
    parts = []

    for doc in search_results:
        parts.append(
            f"""
FILE: {doc['filename']}

{doc['content']}
"""
        )

    return "\n".join(parts)

In [19]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [20]:
def rag(query):
    search_results = search(query)

    context = build_context(search_results)

    prompt = f"""
QUESTION:
{query}

CONTEXT:
{context}
"""

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return response

In [21]:
query = "How does the agentic loop keep calling the model until it stops?"

response = rag(query)

print(response.output_text)
print(response.usage)

It keeps calling the model with a `while True` loop, and it stops only when the model returns a response with **no function calls**.

The core idea is:

1. Send the full message history to the model.
2. If the model asks for a tool call, run the tool.
3. Append the tool result to the history.
4. Call the model again.
5. Repeat until `has_function_calls` is `False`.

In the lesson, the exit condition is:

```python
if has_function_calls == False:
    break
```

So the loop doesn’t decide in advance how many times to call the model. The **model decides** whether it needs another tool call, and your code keeps looping until it finally produces a plain assistant message with no more tool calls.

A simplified version is:

```python
while True:
    response = client.responses.create(...)

    has_function_calls = False

    for item in response.output:
        if item.type == "function_call":
            run_tool(item)
            has_function_calls = True

    if not has_function_calls:
   

In [23]:
from gitsource import chunk_documents

In [24]:
chunked_docs = chunk_documents(
    documents,
    size=2000,
    step=1000
)

In [25]:
len(chunked_docs)

295

In [26]:
chunked_docs[0].keys()

dict_keys(['start', 'content', 'filename'])

In [27]:
from minsearch import Index

chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunked_docs)

In [28]:
def chunk_search(query):
    return chunk_index.search(
        query=query,
        num_results=5
    )

In [29]:
def build_chunk_context(search_results):
    parts = []

    for doc in search_results:
        parts.append(
            f"""
FILE: {doc['filename']}
START: {doc['start']}

{doc['content']}
"""
        )

    return "\n".join(parts)

In [30]:
def chunk_rag(query):

    search_results = chunk_search(query)

    context = build_chunk_context(search_results)

    prompt = f"""
QUESTION:
{query}

CONTEXT:
{context}
"""

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return response

In [31]:
query = (
    "How does the agentic loop keep "
    "calling the model until it stops?"
)

response = chunk_rag(query)

print(response.output_text)
print(response.usage)

The loop keeps calling the model with `while True`, and it only stops when the model makes **no function calls** in a turn.

### How it works
1. Send the current `messages` to the model.
2. Inspect `response.output`.
3. If there’s a `function_call`, run the tool, append the tool result, and set `has_function_calls = True`.
4. If there’s only a normal `message`, store it as the answer.
5. After the turn, check:
   ```python
   if has_function_calls == False:
       break
   ```
   That means the model is done, so the loop exits.

### In plain English
- **Model asks for a tool** → keep looping.
- **Model gives a final answer with no tool calls** → stop looping.

### Why this works
The model decides whether it needs more searching. Your code just:
- executes the tool calls,
- feeds results back,
- repeats until no more tools are requested.

If you want, I can also explain why `messages.extend(response.output)` is important in this loop.
ResponseUsage(input_tokens=2287, input_tokens_detail

In [33]:
response.usage.input_tokens

2287

In [68]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

In [69]:
search_counter = 0

def search_lessons(query: str) -> str:
    """
    Search the lesson chunks for content relevant to the query.
    """
    global search_counter
    search_counter += 1

    results = chunk_index.search(
        query=query,
        num_results=5
    )

    docs = []
    for doc in results:
        docs.append(
            f"""
FILE: {doc['filename']}

{doc['content']}
"""
        )

    return "\n".join(docs)

In [70]:
tools = Tools()
tools.add_tool(search_lessons)

In [71]:
instructions = """
You're a course teaching assistant. Answer the student's question using the
search tool. Make multiple searches with different keywords before answering.
"""

In [72]:
llm_client = OpenAIClient(
    model="gpt-5.4-mini"
)

chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [73]:
question = """
How does the agentic loop work,
and how is it different from plain RAG?
"""

search_counter = 0

result = runner.loop(prompt=question)
print(result.last_message)

The **agentic loop** is basically a repeated cycle:

1. Send the user question plus the conversation history to the LLM
2. Let the LLM decide whether to:
   - answer directly, or
   - call a tool/function
3. If it calls a tool, your code runs that tool
4. Send the tool result back to the LLM
5. Repeat until the model returns a final answer with no more tool calls

The lesson describes it as a `while True` loop that keeps going until there are **no function calls** in the model output. In other words, the model is “in the driver’s seat,” and the loop keeps the agent working until it’s done.

### How that differs from plain RAG

**Plain RAG** is usually a **fixed, one-pass pipeline**:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

So plain RAG does:

- one retrieval step
- one prompt construction step
- one LLM call
- one answer

There’s no iterative decision-making. The retriev

In [74]:
print(search_counter)

4
